In [0]:
-- Game - Goalie Stats
-- Shots faced, saves and save % for each goalie who played in one game, split by situation.
-- Save % is from play-by-play shots on goal, so it can differ slightly from the official figure.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    2026020268 AS game_id   -- e.g. 2025020001 (season 2025, 02 = regular season, game 0001); see Game - Find Game ID
),
game_players AS (                        -- everyone dressed in the game (game rosters)
  SELECT r.game_id, r.player_id, concat_ws(' ', r.first_name, r.last_name) AS full_name,
         r.position_code, r.sweater_number, r.team_id, t.abbrev AS team,
         r.team_id = g.home_team_id AS is_home, g.home_team_id
  FROM nhl.silver.game_rosters r
  JOIN nhl.gold.dim_game g ON r.game_id = g.game_id
  JOIN nhl.gold.dim_team t ON r.team_id = t.team_id
  CROSS JOIN params pr
  WHERE r.game_id = pr.game_id
),
goalie_shots AS (                        -- every shot on goal each goalie faced
  SELECT gp.team, gp.is_home, gp.full_name, gp.player_id,
         f.type_desc_key = 'goal' AS is_goal,
         IF(f.event_owner_team_id = gp.home_team_id, f.situation_code, reverse(f.situation_code)) AS owner_situation,
         CASE WHEN substr(owner_situation, 1, 1) = '0' THEN 'EN'      -- opponent net empty
              WHEN substr(owner_situation, 3, 1) > substr(owner_situation, 2, 1)
               AND substr(owner_situation, 4, 1) = '1' THEN 'PP'      -- more skaters, own goalie in
              WHEN substr(owner_situation, 3, 1) < substr(owner_situation, 2, 1) THEN 'SH'
              ELSE 'EV' END AS strength
  FROM nhl.gold.fact_event f
  JOIN game_players gp ON f.goalie_in_net_id = gp.player_id
  CROSS JOIN params pr
  WHERE f.game_id = pr.game_id
    AND f.type_desc_key IN ('goal', 'shot-on-goal')
    AND COALESCE(f.period_type, '') <> 'SO'    -- shootout attempts aren't stats
)
SELECT team,
       full_name,
       COUNT(*)                                             AS shots_faced,
       COUNT_IF(is_goal)                                    AS goals_against,
       COUNT_IF(NOT is_goal)                                AS saves,
       ROUND(1 - COUNT_IF(is_goal) / COUNT(*), 3)           AS save_pct,
       COUNT_IF(strength = 'EV')                            AS ev_shots,      -- from the shooter's side
       COUNT_IF(strength = 'EV' AND is_goal)                AS ev_goals,
       COUNT_IF(strength = 'PP')                            AS pk_shots,
       COUNT_IF(strength = 'PP' AND is_goal)                AS pk_goals,
       COUNT_IF(strength = 'SH')                            AS sh_shots,
       COUNT_IF(strength = 'SH' AND is_goal)                AS sh_goals
FROM goalie_shots
GROUP BY team, is_home, full_name, player_id
ORDER BY is_home, shots_faced DESC;